# Model evaluation and comparison

**Data sources used in this notebook:**
- `sudan_results.csv` - current set of all runs
- `ethiopia_results.csv` — Ethiopia/Tigray replication runs

In [59]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from utils.constants import ONSET_END_DATE, ONSET_START_DATE
from utils.data_prep import get_clean_combined_data

sudan = pd.read_csv("evaluation/sudan_results.csv").drop_duplicates(subset="run_id", keep="last")
ethiopia = pd.read_csv("evaluation/ethiopia_results.csv")

print(f"Sudan: {len(sudan)} runs")
print(f"Ethiopia: {len(ethiopia)} runs")

Sudan: 1056 runs
Ethiopia: 32 runs


# 1. Choosing the k escalation threshold
## 1.1 Rejecting k=0.25

`k` controls the escalation threshold — a lower k means a looser threshold (escalation is easier to trigger).

**Issue**

Raw AUPR favours the lowest `k`, but that is misleading for conflict prediction. The model was catching 100% of true positives by default rather than through genuine skill. Domain knowledge also matters here - in conflict forecasting, you want a model that's sensitive to real escalations without flagging every small, ordinary shift in conflict as significant.

Three values of `k` (0.25, 0.5, 1.0) were tested in the initial sweep, measuring both the true onset-window prevalence of escalation and the rate at which the F1-optimal threshold collapsed to predicting positive for nearly every region-month.

`k`=0.25 had the highest prevalence (42.1%) and the highest collapse rate (75.0%), making it a poor definition of true escalation — closer to a coin flip than a meaningful departure from a region's baseline. `k`=0.5 was somewhat better (37.0% prevalence, 57.2% collapse) but still substantially collapsed. This motivated dropping `k`=0.25 from further consideration, and investigating the collapse behaviour more closely — covered in Section 1.2.

In [60]:
prevalence_records = []
for k_test in [0.25, 0.5, 1]:
    model_data, predictor_cols = get_clean_combined_data(
        data_sources=[], k=k_test, event_col="sub_event_type", conflict_only_embeddings=True,
    )
    onset_slice = model_data[
        (model_data["year_month"] >= pd.Period(ONSET_START_DATE, freq="M"))
        & (model_data["year_month"] <= pd.Period(ONSET_END_DATE, freq="M"))
    ]
    n_total = len(onset_slice)
    n_escalations = int(onset_slice["target_escalation"].sum())
    prevalence_records.append({
        "k": k_test,
        "onset_prevalence_pct": round(n_escalations / n_total * 100, 1),
        "n_escalations": n_escalations,
        "n_onset_rows": n_total,
    })
prevalence_df_early = pd.DataFrame(prevalence_records).set_index("k")

sudan_pre_fix = sudan[sudan["threshold_fix_applied"] == False]

collapse_rate_pre_fix = sudan_pre_fix.groupby("k")["onset_recall_class1"].apply(lambda x: (x == 1.0).mean())

k_summary_1_1 = pd.DataFrame({
    "mean_onset_aupr": sudan_pre_fix.groupby("k")["onset_aupr"].mean(),
    "max_onset_aupr": sudan_pre_fix.groupby("k")["onset_aupr"].max(),
    "collapse_rate": collapse_rate_pre_fix,
})
k_summary_1_1 = k_summary_1_1.join(prevalence_df_early[["onset_prevalence_pct"]])
k_summary_1_1.round(3)

INFO:AcledClient:Access token correctly retrieved.
INFO:AcledClient:Reading data from data/acled/acled_sudan.csv.
INFO:Events processing:Data grouped by sub_event_type
INFO:Events processing:Escalation target set at 0.25 standard deviations above the mean.
INFO:Data preparation:ACLED data processed.
INFO:AcledClient:Access token correctly retrieved.
INFO:AcledClient:Reading data from data/acled/acled_sudan.csv.
INFO:Events processing:Data grouped by sub_event_type
INFO:Events processing:Escalation target set at 0.5 standard deviations above the mean.
INFO:Data preparation:ACLED data processed.
INFO:AcledClient:Access token correctly retrieved.
INFO:AcledClient:Reading data from data/acled/acled_sudan.csv.
INFO:Events processing:Data grouped by sub_event_type
INFO:Events processing:Escalation target set at 1 standard deviations above the mean.
INFO:Data preparation:ACLED data processed.


,mean_onset_aupr,max_onset_aupr,collapse_rate,onset_prevalence_pct
k,,,,
0.25,0.440,0.526,0.750,42.1
0.50,0.369,0.458,0.662,37.0
1.00,0.341,0.421,0.338,30.6


In [61]:
ks = k_summary_1_1.index.astype(str)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Positive-class prevalence by k", "Threshold-collapse rate (%)"))

fig.add_trace(
    go.Bar(x=ks, y=k_summary_1_1["onset_prevalence_pct"], marker_color="#898781", name="Prevalence"),
    row=1, col=1,
)
fig.add_trace(
    go.Bar(x=ks, y=k_summary_1_1["collapse_rate"] * 100, marker_color="#c0392b", name="Collapse rate"),
    row=1, col=2,
)

fig.update_xaxes(title_text="k", row=1, col=1)
fig.update_xaxes(title_text="k", row=1, col=2)
fig.update_yaxes(title_text="% of onset rows that were escalations", row=1, col=1)
fig.update_yaxes(title_text="% of runs with recall=1.0", row=1, col=2)

fig.update_layout(
    showlegend=False, width=900, height=450,
    plot_bgcolor="white", paper_bgcolor="white",
    margin=dict(t=120),
    title={
        "text": "k=0.25 had the highest prevalence and the highest collapse rate,<br>before the threshold-collapse fix was applied",
        "y": 0.9, "yanchor": "top",
    },
)
fig.show()

## 1.2 The threshold-collapse bug and fix

**Bug**

In the initial set of runs, `optimal_threshold = thresholds[np.argmax(f1_scores)]` picked the lowest tied threshold whenever F1 plateaued, producing "predict everything positive" models. This was discovered because many runs had recall=1 and precision equal to the actual proportion of escalations in the data.

**Fix**

Instead, the model picks the highest threshold among those tied for the best F1 score, making it more conservative and less likely to default to predicting everything positive.

```python
max_f1 = f1_scores.max()
tied_indices = np.flatnonzero(f1_scores == max_f1)
optimal_threshold = thresholds[tied_indices[-1]]
```

**Effect of the fix**

Holding `k` constant (0.5 and 1.0 - as 0.25 was dropped), the collapse rate fell from 50.0% to 40.6%. This is an improvement, but it did not eliminate the issue at the looser threshold. Therefore further work was required to investigate raising `k`, covered in Section 1.3.

In [62]:
pre_threshold_fix = sudan[sudan["threshold_fix_applied"] == False]
threshold_fix = sudan[sudan["threshold_fix_applied"] == True]

pre_fix_shared = pre_threshold_fix[pre_threshold_fix["k"].isin([0.5, 1.0])]
post_fix_shared = threshold_fix[threshold_fix["k"].isin([0.5, 1.0])]

pre_fix_collapse_rate = (pre_fix_shared["onset_recall_class1"] == 1.0).mean()
post_fix_collapse_rate = (post_fix_shared["onset_recall_class1"] == 1.0).mean()

print(f"Pre-fix collapse rate (k=0.5/1.0 only): {pre_fix_collapse_rate:.1%}")
print(f"Post-fix collapse rate (k=0.5/1.0 only): {post_fix_collapse_rate:.1%}")

Pre-fix collapse rate (k=0.5/1.0 only): 50.0%
Post-fix collapse rate (k=0.5/1.0 only): 40.6%


## 1.3 Selecting k among post-fix thresholds

With the corrected threshold logic in place, `k` was tested more broadly (0.5 to 2.5) to find the strictest threshold at which collapse is effectively resolved without giving up genuine model performance.

`k` was chosen once across the whole set of model combinations rather than separately for each one, since testing it against every food, rain, text and event type configuration gives a far larger and more reliable picture than tuning it against a single model would.

`k`=1.75 was chosen mainly because the collapse rate drops to 2.5% at this point, and because a higher `k` pushes the target closer to what escalation actually looks like in practice. Civil conflict onset in the literature is estimated at somewhere between 1.3 and 2.1 events per 100 country-years depending on region (Fearon and Laitin, 2003), so a target that flags conflict as often as 40-70% of the time, which is what we see at the lower `k` values, does not reflect how rare real escalation is. k=1.75 keeps prevalence low without losing model performance. However Fearon and Laitin's figure measures a discrete, national, once-in-a-conflict event, while this model measures a monthly regional deviation from a rolling baseline, which is a naturally more frequent kind of event.

It is worth noting that k=1.65 actually produced a slightly higher onset AUPR (0.4128 versus 0.4043), but it also came with double the collapse rate (5.0% versus 2.5%). Given the small-ish gain and real life esclations rates, 1.75 was chosen. 

In [63]:
prevalence_records = []
for k_test in [0.5, 1.0, 1.25, 1.5, 1.6, 1.65, 1.75, 2.0, 2.5]:
    model_data, predictor_cols = get_clean_combined_data(
        data_sources=[], k=k_test, event_col="sub_event_type", conflict_only_embeddings=True,
    )
    onset_slice = model_data[
        (model_data["year_month"] >= pd.Period(ONSET_START_DATE, freq="M"))
        & (model_data["year_month"] <= pd.Period(ONSET_END_DATE, freq="M"))
    ]
    n_total = len(onset_slice)
    n_escalations = int(onset_slice["target_escalation"].sum())
    prevalence_records.append({
        "k": k_test,
        "onset_prevalence_pct": round(n_escalations / n_total * 100, 1),
        "n_escalations": n_escalations,
        "n_onset_rows": n_total,
    })
prevalence_df_full = pd.DataFrame(prevalence_records).set_index("k")

INFO:AcledClient:Access token correctly retrieved.
INFO:AcledClient:Reading data from data/acled/acled_sudan.csv.
INFO:Events processing:Data grouped by sub_event_type
INFO:Events processing:Escalation target set at 0.5 standard deviations above the mean.
INFO:Data preparation:ACLED data processed.
INFO:AcledClient:Access token correctly retrieved.
INFO:AcledClient:Reading data from data/acled/acled_sudan.csv.
INFO:Events processing:Data grouped by sub_event_type
INFO:Events processing:Escalation target set at 1.0 standard deviations above the mean.
INFO:Data preparation:ACLED data processed.
INFO:AcledClient:Access token correctly retrieved.
INFO:AcledClient:Reading data from data/acled/acled_sudan.csv.
INFO:Events processing:Data grouped by sub_event_type
INFO:Events processing:Escalation target set at 1.25 standard deviations above the mean.
INFO:Data preparation:ACLED data processed.
INFO:AcledClient:Access token correctly retrieved.
INFO:AcledClient:Reading data from data/acled/ac

In [64]:
threshold_fix = sudan[sudan["threshold_fix_applied"] == True]
genuine = threshold_fix[threshold_fix["onset_recall_class1"] <= 0.9] # Where the model is being at least a bit selective

collapse_rate = threshold_fix.groupby("k")["onset_recall_class1"].apply(lambda x: (x > 0.9).mean())

k_summary_1_3 = pd.DataFrame({
    "mean_onset_aupr": threshold_fix.groupby("k")["onset_aupr"].mean(),
    "best_genuine_onset_aupr": genuine.groupby("k")["onset_aupr"].max(),
    "collapse_rate": collapse_rate,
})
k_summary_1_3 = k_summary_1_3.join(prevalence_df_full[["onset_prevalence_pct"]])
k_summary_1_3.round(3)

,mean_onset_aupr,best_genuine_onset_aupr,collapse_rate,onset_prevalence_pct
k,,,,
0.50,0.364,0.412,0.711,37.0
1.00,0.332,0.421,0.445,30.6
1.25,0.334,0.402,0.162,28.7
1.50,0.332,0.399,0.075,24.5
1.60,0.334,0.396,0.075,24.1
1.65,0.333,0.413,0.050,23.6
1.75,0.328,0.404,0.025,22.7
2.00,0.306,0.374,0.025,20.4
2.50,0.242,0.336,0.000,14.4


In [65]:
ks = k_summary_1_3.index.astype(str)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Threshold-collapse rate (%)", "Best genuine onset AUPR"))

fig.add_trace(
    go.Bar(x=ks, y=k_summary_1_3["collapse_rate"] * 100, marker_color="#c0392b", name="Collapse rate"),
    row=1, col=1,
)
fig.add_trace(
    go.Bar(x=ks, y=k_summary_1_3["best_genuine_onset_aupr"], marker_color="#2a78d6", name="Best genuine AUPR"),
    row=1, col=2,
)

fig.update_xaxes(title_text="k", row=1, col=1)
fig.update_xaxes(title_text="k", row=1, col=2)
fig.update_yaxes(title_text="% of runs with recall>0.9", row=1, col=1)
fig.update_yaxes(title_text="onset AUPR", row=1, col=2)

fig.update_layout(
    showlegend=False, width=900, height=450,
    plot_bgcolor="white", paper_bgcolor="white",
    margin=dict(t=120),
    title={
        "text": "Post-fix: collapse falls as k increases, while higher AUPR performance is<br>retained at k=1.75",
        "y": 0.9, "yanchor": "top",
    },
)
fig.show()